In [7]:
import numpy as np
import pandas as pd 
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split

## load dataset

In [8]:
data = pd.read_csv("Dataset/kidney_disease_dataset.csv")
data.head()

,Age,Creatinine_Level,BUN,Diabetes,Hypertension,GFR,Urine_Output,CKD_Status,Dialysis_Needed
0,71,0.30,40.9,0,1,46.8,1622.0,1,0
1,34,1.79,17.1,0,0,43.8,1428.0,1,0
2,80,2.67,15.0,0,1,78.2,1015.0,1,0
3,40,0.97,31.1,0,1,92.8,1276.0,1,0
4,43,2.05,22.8,1,1,62.2,1154.0,0,0


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2304 entries, 0 to 2303
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               2304 non-null   int64  
 1   Creatinine_Level  2304 non-null   float64
 2   BUN               2304 non-null   float64
 3   Diabetes          2304 non-null   int64  
 4   Hypertension      2304 non-null   int64  
 5   GFR               2304 non-null   float64
 6   Urine_Output      2304 non-null   float64
 7   CKD_Status        2304 non-null   int64  
 8   Dialysis_Needed   2304 non-null   int64  
dtypes: float64(4), int64(5)
memory usage: 162.1 KB


## checking for null values

In [10]:
data.isnull().sum()

Age                 0
Creatinine_Level    0
BUN                 0
Diabetes            0
Hypertension        0
GFR                 0
Urine_Output        0
CKD_Status          0
Dialysis_Needed     0
dtype: int64

## split dataset for train and test

In [11]:
X = data.drop(["CKD_Status","Dialysis_Needed"], axis=1)
y = data["CKD_Status"]


In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.4,random_state=43)

## scale dataset

In [ ]:
import joblib
scaler = StandardScaler()
scaler.fit_transform(X_train)
joblib.dump(scaler,"scaler.pkl")

## build model

In [14]:
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard    
import datetime

In [15]:
model = Sequential([
    Dense(units=len(data.columns)-2,activation="relu",input_shape=(X_train.shape[1],)),
    Dense(units=16,activation="relu"),
    Dense(units=8,activation="relu"),
    Dense(units=1,activation="sigmoid")]
)

/Users/varunbhogayta/AI_ML and Data Science/Machine Learning/projects/ANN project/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 7)              │            56 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 329 (1.29 KB)

 Trainable params: 329 (1.29 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from tensorflow.keras.optimizers import Adam

opt = Adam(learning_rate=0.0001)
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [18]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
# history = model.fit(
#     X_train, y_train,
#     validation_data=(X_test, y_test),
#     epochs=1000,
#     callbacks=[tensorboard_callback,early_stopping_callback]
# )

Epoch 1/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4537 - loss: 0.8868 - val_accuracy: 0.4328 - val_loss: 0.9288
Epoch 2/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - loss: 0.8608 - val_accuracy: 0.4349 - val_loss: 0.8984
Epoch 3/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4674 - loss: 0.8374 - val_accuracy: 0.4360 - val_loss: 0.8714
Epoch 4/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4747 - loss: 0.8153 - val_accuracy: 0.4447 - val_loss: 0.8460
Epoch 5/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4783 - loss: 0.7946 - val_accuracy: 0.4588 - val_loss: 0.8214
Epoch 6/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4884 - loss: 0.7753 - val_accuracy: 0.4664 - val_loss: 0.7985
Epoch 7/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4920 - loss: 0.7573 - val_accuracy: 0.4729 - val_loss: 0.7772
Epoch 8/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5000 - loss: 0.7401 - val_accuracy: 0.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = StandardScaler()

accs, rocs, precs, recs, f1s = [], [], [], [], []

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Scale features
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Build and train ANN
    model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)

    # Predictions
    y_pred_proba = model.predict(X_test).ravel()
    y_pred = (y_pred_proba > 0.5).astype(int)

    # Metrics
    accs.append(accuracy_score(y_test, y_pred))
    rocs.append(roc_auc_score(y_test, y_pred_proba))
    precs.append(precision_score(y_test, y_pred))
    recs.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))

print("Accuracy:", np.mean(accs))
print("ROC-AUC:", np.mean(rocs))
print("Precision:", np.mean(precs))
print("Recall:", np.mean(recs))
print("F1:", np.mean(f1s))


Epoch 1/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9647 - loss: 0.0840 - val_accuracy: 0.9566 - val_loss: 0.0899
Epoch 2/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9653 - loss: 0.0824 - val_accuracy: 0.9566 - val_loss: 0.0878
Epoch 3/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9647 - loss: 0.0815 - val_accuracy: 0.9588 - val_loss: 0.0866
Epoch 4/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9664 - loss: 0.0809 - val_accuracy: 0.9610 - val_loss: 0.0860
Epoch 5/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9664 - loss: 0.0805 - val_accuracy: 0.9610 - val_loss: 0.0852
Epoch 6/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9658 - loss: 0.0801 - val_accuracy: 0.9610 - val_loss: 0.0850
Epoch 7/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9669 - loss: 0.0797 - val_accuracy: 0.9610 - val_loss: 0.0850
Epoch 8/1000
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9669 - loss: 0.0797 - val_accuracy: 0.

In [21]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 14393), started 10:11:54 ago. (Use '!kill 14393' to kill it.)

In [22]:
model.save("ckd_ann_model.h5")

array([[ 0.83508458, -1.26970271,  2.11237617, ...,  1.01200261,
        -0.89513895,  0.63578719],
       [-0.984484  ,  0.61192984, -0.14912758, ..., -0.98813974,
        -1.01697031,  0.2388726 ],
       [ 1.27768234,  1.72322959, -0.34867203, ...,  1.01200261,
         0.38002927, -0.60610536],
       ...,
       [-0.88612894, -0.17103136, -1.29888369, ..., -0.98813974,
         1.22066564,  0.9938287 ],
       [ 1.72028011, -1.26970271, -1.29888369, ..., -0.98813974,
        -0.86671163,  1.72832528],
       [ 1.47439246, -1.26970271, -1.080335  , ..., -0.98813974,
         2.0775462 ,  0.08542624]])

['scaler.pkl']